In [20]:
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

In [21]:
!pip install -q -U keras keras-nlp

In [22]:
import keras
import keras_nlp
from IPython.display import display, Markdown, Latex

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

In [23]:
gemma_model_id = "gemma_1.1_instruct_2b_en"
gemma = keras_nlp.models.GemmaCausalLM.from_preset(gemma_model_id)

In [24]:
from typing import List
from IPython.display import display, Markdown, Latex

def convert_message_to_prompt(message: str, model_prefix: str = "") -> str:
    """Converts a message to a prompt for a large language model.

    Args:
        message: The message to convert (str).
        model_prefix: An optional prefix to prepend to the model response (str).

    Returns:
        A string containing the prompt for the large language model (str).
    """

    return (
        f"<start_of_turn>user\n{message}<end_of_turn>\n"
        f"<start_of_turn>model\n{model_prefix}"
    )
print(convert_message_to_prompt("Who are you?"))

<start_of_turn>user
Who are you?<end_of_turn>
<start_of_turn>model



In [25]:
prompt = """Classify the text into neutral, negative, or positive.
Generate only the class, nothing else.
Text: I think the food was awesome."""

prompt = convert_message_to_prompt(prompt)
response = gemma.generate(prompt, max_length=64)
print(response[len(prompt) :])

Positive


In [26]:
prompt = """Generate a single line of hashtags for the given topic by in the same style as the following examples:

Topic: Books
#BooksLover #Books #MyBooks #BestBook #BookOfTheYear

Topic: Games
#GamesLover #Games #MyGames #BestGame #GameOfTheYear

Topic: Movies
"""

prompt = convert_message_to_prompt(prompt)
response = gemma.generate(prompt, max_length=128)
print(response[len(prompt) :])

#MovieLover #Movies #MyMovies #BestMovie #MovieOfTheYear


In [27]:
prompt = """I baked 15 cupcakes for a bake sale. I wanted to share some
with my friends so I gave 5 to my friends. Needing a few more
for taste testing, I baked 4 more cupcakes.
Later, I couldn't resist and a I ate  1 cupcake by myself.
How many cupcakes do I have right now?"""

prompt = convert_message_to_prompt(prompt)
response = gemma.generate(prompt, max_length=512)
print(response[len(prompt) :])

You have 15 cupcakes. You gave 5 to your friends, so you have 15 - 5 = 10 cupcakes left. You baked 4 more cupcakes, so you have 10 + 4 = 14 cupcakes right now.


In [28]:
prompt = """I baked 15 cupcakes for a bake sale. I wanted to share some
with my friends so I gave 5 to my friends. Needing a few more
for taste testing, I baked 4 more cupcakes.
Later, I couldn't resist and a I ate 1 cupcake by myself.
How many cupcakes do I have right now? Explain your thinking step by step
including the number of cupcakes per step."""

prompt = convert_message_to_prompt(prompt)
response = gemma.generate(prompt, max_length=512)
display(Markdown(response[len(prompt) :]))

**Step 1:** After sharing with friends, you have 10 cupcakes left.


**Step 2:** You bake 4 more cupcakes, so you have 14 cupcakes in total.


**Step 3:** You eat 1 cupcake, leaving you with 13 cupcakes.

In [29]:
prompt = """Which is heavier? 1kg of feathers or 1kg of stones?"""

prompt = convert_message_to_prompt(prompt)
response = gemma.generate(prompt, max_length=512)
display(Markdown(response[len(prompt) :]))

**1kg of feathers is lighter than 1kg of stones.**

This is because feathers are much lighter than stones.

In [30]:
prompt = """Simulate a multi turn debate of three experts.
They need to answer the following question: Which is heavier? 1kg of feathers or 1kg of stones?
They need to debate in rounds and provide explaination until they reach the same conclusion."""

prompt = convert_message_to_prompt(prompt)
response = gemma.generate(prompt, max_length=512)
display(Markdown(response[len(prompt) :]))

**Round 1:**

**Expert 1:** The feathers are significantly lighter. They are composed primarily of air, while stones are dense and have a substantial amount of mass associated with their size and composition.

**Expert 2:** While feathers are indeed much lighter, it's important to consider the surface area of the objects. Feathers have a large surface area-to-mass ratio, making them relatively easy to lift and move.

**Expert 3:** True, but the surface area of the stones also plays a role. The larger the stone, the greater its surface area, and the heavier it will feel.

**Round 2:**

**Expert 1:** The surface area argument is valid, but it's crucial to note that the weight of an object is determined by its mass and the force of gravity acting on it.

**Expert 2:** While mass is a fundamental property of an object, it's the force of gravity that determines its weight. The mass of 1kg of feathers is the same as the weight of 1kg of stones, regardless of their shape or surface area.

**Expert 3:** You're right. The force of gravity pulling the objects down is the same. However, the feathers' surface area allows them to interact more with the air, creating a resistance that makes them feel lighter.

**Round 3:**

**Expert 1:** The resistance due to surface area is a valid point, but it's important to consider the overall context.

**Expert 2:** The weight is determined by the force of gravity and the mass. In this case, the force of gravity is the same for both objects.

**Expert 3:** True, but the feathers' surface area creates a significant additional force that makes them feel lighter.

**Conclusion:**

After three rounds of debate, all three experts agree that the weight of 1kg of feathers and 1kg of stones is the same. The force of gravity pulling them down is the same, but the feathers' surface area creates a resistance that makes them feel lighter.

In [31]:
prompt_1 = """
Extract all city names from the following text:

The aroma of fresh bread wafted through the Paris market, tempting Amelia
as she hurried to catch her flight to Tokyo. She dreamt of indulging in steaming
ramen after a whirlwind tour of ancient temples. Back home in Chicago,
she'd recount her adventures, photos filled with Eiffel Tower selfies
and neon-lit Tokyo nights.
"""
prompt_1 = convert_message_to_prompt(prompt_1)
prompt_1_response = gemma.generate(prompt_1, max_length=512)

print("--- After first prompt: ---")
display(Markdown(prompt_1_response[len(prompt_1) :]))

--- After first prompt: ---


- Paris
- Tokyo
- Chicago

In [32]:
previous_response = {prompt_1_response[len(prompt_1) :]}

prompt_2 = f"""Convert the following cities into a valid Python list.
Make it uppercase and remove all unecessary characters:\n{previous_response}"""
prompt_2 = convert_message_to_prompt(prompt_2)
prompt_2_response = gemma.generate(prompt_2, max_length=512)

print("--- After second prompt: ---")
display(Markdown(prompt_2_response[len(prompt_2) :]))

--- After second prompt: ---


```python
cities = ['PARIS', 'TOKYO', 'CHICAGO']
```

In [33]:
import random
import time
from datetime import datetime

def get_weather_data():
    print("(py function) fetching the current weather")
    return random.choice(["Rainy", "Sunny", "Snowing"])


def get_today_date():
    print("(py function) fetching the current date")
    return datetime.today().strftime("%Y-%m-%d")


def get_time():
    print("(py function) fetching the current time")
    return datetime.today().strftime("%H:%M:%S")

In [34]:
mapping = {
    "get_weather_data": get_weather_data,
    "get_today_date": get_today_date,
    "get_time": get_time,
}

In [35]:
def ask_model_with_fn_calling(question: str) -> str:
    prompt = f"""Your job is to select the best tool for the given query. Only respond with the name of the tool.
  When you get tool output, format the output message to answer user question.

  Available tools:
  - get_weather_data: Checks the current weather
  - get_today_date: Retruns today's date
  - get_time: Returns current time in local timezone

  Examples:
  User: What's the weather?
  Tool: get_weather_data

  User: What's the time?
  Tool: get_time
  Tool output: 21:00
  Answer: The current time is: 21:00

  User: Give me today's date
  Tool: get_today_date
  Tool output: 21-10-2024
  Answer: Today's date is 21-10-2024

  User: What's the weather?
  Tool: get_weather_data
  Tool output: Rainy
  Answer: It's rainy today.

  User: {question}"""

    prompt = convert_message_to_prompt(prompt, "Tool:")
    response = gemma.generate(prompt, max_length=512)

    command = lines = response.split("\n")[-1]
    tool_name = command.split(" ")[-1]

    try:
        output = mapping[tool_name]()
    except KeyError:
        print(f"Model decided to use function called '{tool_name}' which doesnt exist.")
        return

    new_prompt = response + f"\nTool output: {output}\nAnswer: "
    new_response = gemma.generate(new_prompt, max_length=512)
    print(new_response[len(new_prompt) :])

In [36]:
ask_model_with_fn_calling("What's the weather today?")

(py function) fetching the current weather

It's rainy today.


In [37]:
ask_model_with_fn_calling("What's the time?")

(py function) fetching the current time
09:45:55


In [38]:
ask_model_with_fn_calling("Answer with today's date")

(py function) fetching the current date
2026-06-05
